# 基于推荐模型的 GMV 归因算法设计

本 Notebook 将演示如何基于 DIN 和 MMOE 模型设计 GMV (Gross Merchandise Value) 归因算法。
我们将从两个维度进行归因：

1.  **特征归因 (Feature Attribution)**：在单次推荐中，用户的哪些历史行为促成了这次高 GMV 的转化？
    *   **方法**：提取 DIN 模型的 **Attention Weights**。
2.  **数据估值 (Data Valuation)**：在模型训练中，哪些训练样本对提升模型整体的 GMV 预测能力贡献最大？
    *   **方法**：使用 **Data Shapley** 思想（简化版 Leave-One-Out）。

In [1]:
%pip install pandas numpy tensorflow scikit-learn
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.preprocessing.sequence import pad_sequences

# --------------------------
# 1. 数据模拟准备
# --------------------------
NUM_USERS = 100
NUM_ITEMS = 50
EMBEDDING_DIM = 8
MAX_SEQ_LEN = 5

# 模拟一些数据
# 假设 Item ID 对应不同的价格 (Price)
item_prices = {i: np.random.randint(10, 100) for i in range(NUM_ITEMS)}

# 构造样本: (User, Target_Item, History) -> Label (0/1)
X_user = np.random.randint(0, NUM_USERS, 1000)
X_item = np.random.randint(0, NUM_ITEMS, 1000)
X_history = np.random.randint(0, NUM_ITEMS, (1000, MAX_SEQ_LEN))
y_label = np.random.randint(0, 2, 1000)

print("数据准备完成。")

   ---------------------------------------- 0.0/331.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/331.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/331.7 MB ? eta -:--:--
   ---------------------------------------- 2.6/331.7 MB 5.0 MB/s eta 0:01:06
    --------------------------------------- 5.0/331.7 MB 7.0 MB/s eta 0:00:47
   - -------------------------------------- 9.2/331.7 MB 11.0 MB/s eta 0:00:30
   - -------------------------------------- 10.2/331.7 MB 9.2 MB/s eta 0:00:35
   - -------------------------------------- 10.5/331.7 MB 8.4 MB/s eta 0:00:39
   - -------------------------------------- 11.3/331.7 MB 7.6 MB/s eta 0:00:43
   - -------------------------------------- 11.5/331.7 MB 7.4 MB/s eta 0:00:44
   - -------------------------------------- 14.4/331.7 MB 7.4 MB/s eta 0:00:43
   - -------------------------------------- 16.0/331.7 MB 8.0 MB/s eta 0:00:40
   -- ------------------------------------- 18.4/331.7 MB 8.0 MB/s eta 0:00:39


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


数据准备完成。


## 方法一：基于 DIN Attention 的特征归因

**原理**：DIN 模型中的 Local Activation Unit 计算了 `Target Item` 与 `History Items` 的相关性权重。我们可以直接将此权重视为“历史行为对当前推荐结果的贡献度”。

**场景**：用户最终购买了推荐的“波士顿龙虾”（高 GMV）。是因为他之前看过“海鲜”，还是因为他看过“优惠券”？Attention 权重会告诉我们。

In [ ]:
def build_din_with_attention_output():
    # Inputs
    user_input = Input(shape=(1,), name='User_ID')
    item_input = Input(shape=(1,), name='Target_Item_ID')
    history_input = Input(shape=(MAX_SEQ_LEN,), name='History_Item_IDs')

    embedding_layer = layers.Embedding(NUM_ITEMS, EMBEDDING_DIM)
    
    target_emb = embedding_layer(item_input) # (B, 1, E)
    history_emb = embedding_layer(history_input) # (B, T, E)
    
    # --- Attention Mechanism ---
    query = tf.tile(target_emb, [1, MAX_SEQ_LEN, 1])
    # 简单的拼接+MLP计算分数
    concat = layers.Concatenate()([query, history_emb, query * history_emb])
    att_score = layers.Dense(1, activation='sigmoid', name='Attention_Score')(concat) # (B, T, 1)
    
    # Output both Prediction and Attention Weights
    # ... (省略 MLP 部分以简化展示，只展示核心逻辑)
    
    model = Model(inputs=[item_input, history_input], outputs=att_score)
    return model

din_att_model = build_din_with_attention_output()

# 模拟归因
sample_target = np.array([[10]]) # 假设 Target Item ID 10 (价格 $80)
sample_history = np.array([[1, 5, 10, 20, 0]]) # 历史行为，其中包含 ID 10

weights = din_att_model.predict([sample_target, sample_history])
weights = weights[0].flatten()

print(f"Target Item 10 (Price ${item_prices[10]}) 的归因分析:")
for i, item_id in enumerate(sample_history[0]):
    contribution = weights[i] * item_prices[10] # 简单的线性归因：权重 * 最终价值
    print(f"  - History Item {item_id}: Attention Weight = {weights[i]:.4f} -> GMV Contribution approx ${contribution:.2f}")

## 方法二：基于 Data Shapley 的数据估值

**原理**：根据 Shapley Value 理论（如环境中论文所述），一个数据点的价值等于它对联盟效用（Utility）的边际贡献。这里我们将 Utility 定义为 **验证集上的总预测 GMV** (或加权准确率)。

**公式**：`Value(z) = Utility(Dataset + z) - Utility(Dataset)`

**注意**：完整 Shapley 计算需要指数级复杂度。这里演示 **Leave-One-Out (LOO)** 近似，即“去掉这个样本，模型损失了多少 GMV”。

In [ ]:
# 定义 Utility 函数：模型在验证集上产生的 预期 GMV
def calculate_utility(model, val_x, val_y):
    # 预测购买概率 (CVR)
    preds = model.predict(val_x, verbose=0).flatten()
    
    # 计算预期 GMV： Sum(Probability * Price)
    # 这里我们需要知道验证集中每个样本对应的 Target Item 的价格
    total_expected_gmv = 0
    for i, prob in enumerate(preds):
        item_id = val_x[1][i][0] # 假设 val_x[1] 是 Item Input
        price = item_prices.get(item_id, 0)
        # 只有当真实标签为 1 (购买) 时，我们才特别关注模型的准确度带来的收益
        # 或者简单的 Utility = 预测准确度 * 价格
        if val_y[i] == 1:
            total_expected_gmv += prob * price
    return total_expected_gmv

# 模拟简单的 LOO 归因
def data_valuation_loo(model_builder, train_x, train_y, val_x, val_y):
    # 1. 训练全量模型
    full_model = model_builder()
    full_model.compile(optimizer='adam', loss='binary_crossentropy')
    full_model.fit(train_x, train_y, epochs=1, verbose=0)
    base_utility = calculate_utility(full_model, val_x, val_y)
    print(f"Base Utility (Full Data): ${base_utility:.2f}")
    
    contributions = []
    
    # 2. 逐一移除样本 (演示前 5 个)
    for i in range(5):
        # 移除第 i 个样本
        sub_x = [np.delete(x, i, axis=0) for x in train_x]
        sub_y = np.delete(train_y, i, axis=0)
        
        # 重新训练 (实际中会使用 Influence Function 近似，避免重训)
        sub_model = model_builder()
        sub_model.compile(optimizer='adam', loss='binary_crossentropy')
        sub_model.fit(sub_x, sub_y, epochs=1, verbose=0)
        
        sub_utility = calculate_utility(sub_model, val_x, val_y)
        
        # 边际贡献 = 全量效用 - 移除后效用
        diff = base_utility - sub_utility
        contributions.append(diff)
        print(f"Data Point {i}: Marginal Contribution to GMV = ${diff:.4f}")
        
    return contributions

# 构建一个简单的 MLP 模型用于演示
def build_simple_model():
    u = Input(shape=(1,))
    i = Input(shape=(1,))
    h = Input(shape=(MAX_SEQ_LEN,))
    x = layers.Concatenate()([u, i, layers.Flatten()(h)])
    out = layers.Dense(1, activation='sigmoid')(x)
    return Model([u, i, h], out)

# 运行演示
# 注意：为了演示代码可运行，这里只用了极少量数据
_ = data_valuation_loo(build_simple_model, 
                       [X_user[:20], X_item[:20], X_history[:20]], y_label[:20], 
                       [X_user[20:30], X_item[20:30], X_history[20:30]], y_label[20:30])